In [ ]:
import polars as pl

# Define column names
columns = [
    "time",
    "src_user",
    "dst_user",
    "src_computer",
    "dst_computer",
    "auth_type",
    "logon_type",
    "auth_orientation",
    "status"
]

# Lazy scan (does NOT load into memory)
df = (
    pl.scan_csv(
        "data_windows/auth.txt",
        has_header=False,
        new_columns=columns,
        infer_schema_length=1000
    )
    # Convert time to integer
    .with_columns(
        pl.col("time").cast(pl.Int64)
    )
    # Filter ONLY redteam window
    .filter(
        (pl.col("time") >= 150885) &
        (pl.col("time") <= 2557047)
    )
    # Optional: drop useless columns for now
    .select([
        "time",
        "src_user",
        "src_computer",
        "dst_computer",
        "logon_type",
        "status"
    ])
)

# Write to parquet WITHOUT loading entire dataset
df.sink_parquet("filtered_auth.parquet")


In [ ]:
import os
print(os.path.getsize("filtered_auth.parquet") / 1e9, "GB")


In [ ]:
import polars as pl

auth = pl.read_parquet("filtered_auth.parquet")

print(auth.head())
print(auth.shape)


In [ ]:
red = pl.read_csv(
    "data_windows/redteam.txt.gz",
    has_header=False,
    new_columns=["time", "user", "src_computer", "dst_computer"]
)

print(red.head())
print(red.shape)


FILTERING ONLY THE REAL USERS

In [1]:
import polars as pl

auth = pl.scan_parquet("filtered_auth.parquet")


In [2]:
auth_users = auth.filter(
    pl.col("src_user").str.starts_with("U")
).rename({
    "src_user": "user"
})


In [5]:
import polars as pl

red = pl.scan_csv(
    "data_windows/redteam.txt.gz",
    has_header=False,
    new_columns=["time", "user", "src_computer", "dst_computer"],
    infer_schema_length=1000
)


In [6]:
labeled = auth_users.join(
    red,
    on=["time", "user", "src_computer", "dst_computer"],
    how="left"
)


In [7]:
labeled = labeled.with_columns(
    pl.when(pl.col("user_right").is_not_null())
      .then(1)
      .otherwise(0)
      .alias("is_compromise")
)


In [8]:
summary = labeled.select([
    pl.count().alias("total_events"),
    pl.col("is_compromise").sum().alias("compromise_events")
]).collect()

print(summary)

C:\Users\kruti\AppData\Local\Temp\ipykernel_17324\1967538464.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total_events"),


ColumnNotFoundError: unable to find column "user_right"; valid columns: ["time", "user", "src_computer", "dst_computer", "logon_type", "status"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
LEFT JOIN:
LEFT PLAN ON: [col("time"), col("user"), col("src_computer"), col("dst_computer")]
  SELECT [col("time"), col("src_user").alias("user"), col("src_computer"), col("dst_computer"), col("logon_type"), col("status")]
    FILTER col("src_user").str.starts_with(["U"])
    FROM
      Parquet SCAN [filtered_auth.parquet]
      PROJECT */6 COLUMNS
      ESTIMATED ROWS: 472067100
RIGHT PLAN ON: [col("time"), col("user"), col("src_computer"), col("dst_computer")]
  Csv SCAN [data_windows/redteam.txt.gz]
  PROJECT */4 COLUMNS
END LEFT JOIN

In [2]:
import polars as pl

# 1️⃣ Lazy load auth
auth = pl.scan_parquet("filtered_auth.parquet")

auth_users = (
    auth
    .filter(pl.col("src_user").str.starts_with("U"))
    .rename({"src_user": "user"})
)

# 2️⃣ Lazy load redteam and ADD marker column
red = (
    pl.scan_csv(
        "data_windows/redteam.txt.gz",
        has_header=False,
        new_columns=["time", "user", "src_computer", "dst_computer"]
    )
    .with_columns(
        pl.lit(1).alias("red_flag")
    )
)

# 3️⃣ Join
labeled = auth_users.join(
    red,
    on=["time", "user", "src_computer", "dst_computer"],
    how="left"
)

# 4️⃣ Create binary label
labeled = labeled.with_columns(
    pl.when(pl.col("red_flag") == 1)
      .then(1)
      .otherwise(0)
      .alias("is_compromise")
)

# 5️⃣ Collect summary only
summary = labeled.select([
    pl.count().alias("total_events"),
    pl.col("is_compromise").sum().alias("compromise_events")
]).collect()

print(summary)


C:\Users\kruti\AppData\Local\Temp\ipykernel_28216\3316299865.py:41: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total_events"),


shape: (1, 2)
┌──────────────┬───────────────────┐
│ total_events ┆ compromise_events │
│ ---          ┆ ---               │
│ u32          ┆ i32               │
╞══════════════╪═══════════════════╡
│ 159154450    ┆ 739               │
└──────────────┴───────────────────┘


In [3]:
comp_samples = (
    labeled
    .filter(pl.col("is_compromise") == 1)
    .select(["time", "user", "src_computer", "dst_computer"])
    .limit(10)
    .collect()
)

print(comp_samples)


shape: (10, 4)
┌────────┬────────────┬──────────────┬──────────────┐
│ time   ┆ user       ┆ src_computer ┆ dst_computer │
│ ---    ┆ ---        ┆ ---          ┆ ---          │
│ i64    ┆ str        ┆ str          ┆ str          │
╞════════╪════════════╪══════════════╪══════════════╡
│ 150885 ┆ U620@DOM1  ┆ C17693       ┆ C1003        │
│ 151036 ┆ U748@DOM1  ┆ C17693       ┆ C305         │
│ 151648 ┆ U748@DOM1  ┆ C17693       ┆ C728         │
│ 151993 ┆ U6115@DOM1 ┆ C17693       ┆ C1173        │
│ 153792 ┆ U636@DOM1  ┆ C17693       ┆ C294         │
│ 155219 ┆ U748@DOM1  ┆ C17693       ┆ C5693        │
│ 155399 ┆ U748@DOM1  ┆ C17693       ┆ C152         │
│ 155460 ┆ U748@DOM1  ┆ C17693       ┆ C2341        │
│ 155591 ┆ U748@DOM1  ┆ C17693       ┆ C332         │
│ 156658 ┆ U748@DOM1  ┆ C17693       ┆ C4280        │
└────────┴────────────┴──────────────┴──────────────┘


In [4]:
stats = (
    labeled
    .select([
        pl.len().alias("total"),
        pl.col("is_compromise").sum().alias("compromised")
    ])
    .collect(streaming=True)
)

print(stats)


C:\Users\kruti\AppData\Local\Temp\ipykernel_28216\1785684761.py:7: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


shape: (1, 2)
┌───────────┬─────────────┐
│ total     ┆ compromised │
│ ---       ┆ ---         │
│ u32       ┆ i32         │
╞═══════════╪═════════════╡
│ 159154450 ┆ 739         │
└───────────┴─────────────┘


In [5]:
total = stats["total"][0]
comp = stats["compromised"][0]
rate = comp / total

print("Total:", total)
print("Compromised:", comp)
print("Rate:", rate)


Total: 159154450
Compromised: 739
Rate: 4.64328832778474e-06


In [6]:
labeled.sink_parquet("labeled_auth.parquet")
